# DBSCAN vs HDBSCAN — 우리 데이터로 직접 눈으로 비교하기

이 노트북은 MASIL 합성 주행 데이터(`gaip_visit_events.csv`, 180명 × 14개월, 80,032건)에서
**같은 운전자 한 명**에게 DBSCAN과 HDBSCAN을 나란히 돌려, 생활권이 어떻게 만들어지는지 그림으로 비교합니다.

**사용법 (Colab)**: 메뉴에서 `런타임 → 모두 실행`. 첫 셀이 GitHub에서 데이터를 자동으로 받아옵니다.

**그림 읽는 법**
- 점 = 기준선(첫 2개월) 방문 좌표 1개. 색 = 알고리즘이 배정한 군집. 회색 × = 노이즈(버려진 방문)
- 점선 원 = 그 군집의 생활권 원 (반경 = max(코어 500m, 방문거리 P90), 상한 2,000m)
- **최종 생활권 = 원들의 합집합.** 판정(생활권 안/밖)은 이 합집합 기준
- `eval coverage` = 이후 12개월 방문 중 이 생활권 안에 들어온 비율


In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("seniorcareservice"):
        !git clone --branch claude/gaip-dashboard-refine --depth 1 https://github.com/summit1123/seniorcareservice.git
    ROOT = "seniorcareservice"
else:
    ROOT = ".."  # 로컬에서 notebooks/ 폴더 기준
sys.path.insert(0, ROOT)
print("데이터 위치:", ROOT + "/data/fixtures/gaip_visit_events.csv")

In [ ]:
import csv, math
from collections import defaultdict
import matplotlib.pyplot as plt
from src.gaip_simulation.clustering import dbscan_distinct_days, haversine_m, percentile_nearest_rank

CSV_PATH = ROOT + "/data/fixtures/gaip_visit_events.csv"
CORE_M, CAP_M = 500.0, 2000.0
COLORS = ["#1D9E75", "#378ADD", "#EF9F27", "#D4537E", "#7F77DD", "#639922", "#D85A30", "#0F6E56"]

with open(CSV_PATH, encoding="utf-8") as f:
    ALL_ROWS = list(csv.DictReader(f))
for r in ALL_ROWS:
    r["latitude"] = float(r["latitude"]); r["longitude"] = float(r["longitude"])
print("전체 방문 이벤트:", len(ALL_ROWS))

def load_driver(driver_id):
    rows = [r for r in ALL_ROWS if r["driver_id"] == driver_id]
    fit = [r for r in rows if r["period_role"] == "baseline"]
    ev = [r for r in rows if r["period_role"] != "baseline"]
    return fit, ev, rows[0]["environment_id"], rows[0]["designed_type"]

def offsets(events, anchor):
    lat0, lon0 = anchor
    return [((e["longitude"] - lon0) * 111_320.0 * math.cos(math.radians(lat0)),
             (e["latitude"] - lat0) * 111_320.0) for e in events]

def zone_clusters(events, labels):
    groups = defaultdict(list)
    for e, l in zip(events, labels):
        if l >= 0:
            groups[l].append(e)
    out = []
    for l, evs in sorted(groups.items()):
        lat = sum(e["latitude"] for e in evs) / len(evs)
        lon = sum(e["longitude"] for e in evs) / len(evs)
        d = [haversine_m(e["latitude"], e["longitude"], lat, lon) for e in evs]
        p90 = percentile_nearest_rank(d, 0.90)
        out.append({"lat": lat, "lon": lon, "r": max(CORE_M, min(p90, CAP_M))})
    return out

def coverage(eval_events, clusters):
    if not clusters or not eval_events:
        return 0.0
    inz = sum(1 for e in eval_events
              if any(haversine_m(e["latitude"], e["longitude"], c["lat"], c["lon"]) <= c["r"]
                     for c in clusters))
    return inz / len(eval_events) * 100

def run_hdbscan(events, anchor, mcs, ms):
    from sklearn.cluster import HDBSCAN
    import numpy as np
    xy = offsets(events, anchor)
    h = HDBSCAN(min_cluster_size=mcs, min_samples=ms, allow_single_cluster=True, copy=True)
    return h.fit_predict(np.array(xy)).tolist()

def compare(driver_id, configs=None):
    if configs is None:
        configs = [
            ("DBSCAN eps=260m + 3 days", "dbscan", (260, 3)),
            ("DBSCAN eps=1100m + 3 days", "dbscan", (1100, 3)),
            ("HDBSCAN mcs=3, ms=2", "hdbscan", (3, 2)),
            ("HDBSCAN mcs=5, ms=3", "hdbscan", (5, 3)),
        ]
    fit, ev, env, dtype = load_driver(driver_id)
    anchor = (sum(e["latitude"] for e in fit) / len(fit),
              sum(e["longitude"] for e in fit) / len(fit))
    n = len(configs)
    ncols = 2
    nrows = (n + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 6 * nrows))
    axes = axes.flat if n > 1 else [axes]
    for ax, (title, algo, params) in zip(axes, configs):
        if algo == "dbscan":
            labels = dbscan_distinct_days(fit, eps_m=params[0], min_distinct_days=params[1])["labels"]
        else:
            labels = run_hdbscan(fit, anchor, *params)
        clusters = zone_clusters(fit, labels)
        cov = coverage(ev, clusters)
        for (x, y), l in zip(offsets(fit, anchor), labels):
            if l < 0:
                ax.scatter(x, y, marker="x", c="#999999", s=42, zorder=3)
            else:
                ax.scatter(x, y, c=COLORS[l % len(COLORS)], s=30, zorder=3, edgecolors="none")
        for i, c in enumerate(clusters):
            cx = (c["lon"] - anchor[1]) * 111_320.0 * math.cos(math.radians(anchor[0]))
            cy = (c["lat"] - anchor[0]) * 111_320.0
            ax.add_patch(plt.Circle((cx, cy), c["r"], fill=True, alpha=0.10,
                                    color=COLORS[i % len(COLORS)], zorder=1))
            ax.add_patch(plt.Circle((cx, cy), c["r"], fill=False, linestyle="--", linewidth=1.4,
                                    color=COLORS[i % len(COLORS)], zorder=2))
        noise = sum(1 for l in labels if l < 0)
        ax.set_title(f"{title}\nclusters={len(clusters)}  noise={noise}  eval coverage={cov:.1f}%",
                     fontsize=11)
        ax.set_aspect("equal"); ax.grid(alpha=0.25)
        ax.set_xlabel("east (m)"); ax.set_ylabel("north (m)")
    fig.suptitle(f"driver {driver_id} ({env}, {dtype}) - baseline visits, zone = union of circles",
                 fontsize=13)
    fig.tight_layout()
    plt.show()

## 운전자 한 명 비교

아래 `DRIVER_ID`를 바꿔가며 실행해 보세요. 유형별 대표 운전자는 다음 셀에서 출력됩니다.

In [ ]:
DRIVER_ID = "gaip-123"   # 광역 · 다생활권형 — 바꿔서 실행해 보세요
compare(DRIVER_ID)

In [ ]:
# 유형 x 환경 대표 운전자 목록
reps = {}
for r in ALL_ROWS:
    key = (r["environment_id"], r["designed_type"])
    reps.setdefault(key, r["driver_id"])
print(f"{'환경':24s} {'유형':26s} 대표 운전자")
for (env, t), d in sorted(reps.items()):
    print(f"{env:24s} {t:26s} {d}")

## 그림에서 확인하게 되는 것

1. **원의 개수는 설정마다 다른데, 원들의 합집합(=최종 생활권)은 거의 같습니다.**
   HDBSCAN 3/2가 거점을 6조각 내도, 조각마다 최소 500m 원이 붙어 합집합이 같은 영역을 덮습니다.
   판정은 합집합 기준이므로 파편화 자체는 상품 손해가 아닙니다.
2. **눈에 보이는 유일한 손실은 DBSCAN eps=1100 (큰 eps)** — 남쪽 거점 두 개가 하나로
   병합되고, 병합된 덩어리의 버퍼가 상한(2,000m)에 걸려 커버리지가 떨어집니다.
   "농촌은 큰 eps"라는 가정의 비용이 이 패널입니다.
3. **HDBSCAN 5/3은 DBSCAN eps=260과 사실상 같은 결과**를 냅니다.
   두 방식의 차이는 성능이 아니라 "왜 이렇게 묶었는지 문장으로 설명 가능한가"입니다.

## 그래서 어느 쪽이 "맞는가"

이 합성 데이터로는 **성능으로 우열을 가릴 수 없습니다** (위 그림이 그 증거입니다 — 결과 존이 거의 같음).
따라서 선택 기준은 성능 밖에 있습니다:

| | A안: DBSCAN(3일 내장) | B안: HDBSCAN + 3일 후필터 |
|---|---|---|
| 판정 규칙을 문장으로 쓸 수 있나 | "반경 ε 안, 서로 다른 3일 방문 = 거점" ✅ | 계층 안정성 기준 — 문장화 어려움 ❌ |
| 공간 스케일의 출처 | 지역 풀링 실측(k-거리)으로 도출 | 개인 데이터에서 자동 (개인당 점 ~100개) |
| 남는 검증 과제 | eps 컷 백분위 | min_samples 민감도 |

**현재 결론**: 발표/운영은 A안(설명 가능, 구현 완료) 유지, B안은 실데이터 파일럿에서
같은 채점표(커버리지·노이즈·거점수·기간 안정성)로 베이크오프하여 사전 등록된 기준으로 결정.

## (보너스) eps를 "정하지 않고 재는" 절차 — k-거리 산포 측정

같은 장소를 **다른 날** 재방문했을 때 좌표가 서로 얼마나 떨어져 찍히는지(주차 산포)를
지역별로 모아 백분위를 보면, eps를 사람이 정하지 않고 데이터에서 도출할 수 있습니다.
실데이터 파일럿에서 이 절차를 그대로 쓰면 됩니다.

In [ ]:
from collections import defaultdict as dd
by_env_driver = dd(lambda: dd(list))
for r in ALL_ROWS:
    if r["period_role"] == "baseline":
        by_env_driver[r["environment_id"]][r["driver_id"]].append(
            (r["latitude"], r["longitude"], r["visit_date"]))

def pct(xs, q):
    xs = sorted(xs)
    return xs[max(0, min(len(xs) - 1, int(len(xs) * q)))]

print(f"{'환경':24s} {'P50':>6s} {'P90':>6s} {'P95':>6s}   시사 eps")
for env, drivers in by_env_driver.items():
    pooled = []
    for d, pts in drivers.items():
        for i, (la, lo, dt) in enumerate(pts):
            ds = sorted(haversine_m(la, lo, la2, lo2)
                        for j, (la2, lo2, dt2) in enumerate(pts) if j != i and dt2 != dt)
            if len(ds) >= 2:
                pooled.append(ds[1])
    p50, p90, p95 = pct(pooled, 0.50), pct(pooled, 0.90), pct(pooled, 0.95)
    print(f"{env:24s} {p50:5.0f}m {p90:5.0f}m {p95:5.0f}m   ~{math.ceil(p95 / 10) * 10}m")
print()
print("합성 데이터의 산포(수십 m)는 실제보다 훨씬 촘촘합니다 — 이 표의 값 자체가 아니라")
print("'절차가 지역별 산포 차이를 복원한다'는 사실이 검증 포인트입니다.")